# Is the Judge the Bottleneck? Kaggle notebook

Run this notebook **top to bottom** at the start of every Kaggle session (Kaggle forgets everything between sessions). It sets the project up, checks every piece against the Mac, then runs the pipeline on the GPU and **logs the run**.

**Before you press Run All, in the right-hand panel (Session options):**
1. **Accelerator:** `GPU T4 x2` (needed for sections 9–11; sections 1–8 also work on CPU).
2. **Internet:** On (needs a phone-verified Kaggle account).
3. **Add Input → Your Datasets → `hotpotqa-bm25-index`** (our saved search index).

Every section stops with a clear message if something goes wrong. Long lists of yellow "dependency conflict" warnings during installs are normal on Kaggle; only a red error that stops a cell matters. A warning about `HF_TOKEN` is harmless too.

**To keep the run logs:** click **Save Version** when you're done. That saves `/kaggle/working/runs/` with the notebook.

## 1. Settings

**What it does:** the only lines you normally change: which GitHub branch to run, which optional parts to run, and which scenario section 11 runs.
**Files used:** none.

In [ ]:
# Here we choose what this session does.
REPO_URL = "https://github.com/shoimya/Is-the-Judge-the-Bottleneck.git"
BRANCH = "SC.V2"                  # the GitHub branch to run

BUILD_INDEX_IF_MISSING = False    # True only if the hotpotqa-bm25-index dataset is lost (download 1.55 GB + ~12 min build)
RUN_MODEL_CHECKS = True           # sections 9-11 need GPU T4 x2: install vLLM, check the model, run the pipeline

# Section 11: which scenario to run, and on what.
SCENARIO = "single-turn"          # working now: "closed-book" or "single-turn" (the rest come in T08)
QUESTION_SET = None               # None = just the first Pilot question; "pilot" = all 100; "test" = all 1,000

## 2. Helpers and machine check

**What it does:** defines `run_or_stop()`, which runs a terminal command and **stops the notebook if it fails** (instead of carrying on into confusing errors). Then it checks this machine: Kaggle or Colab, Python version, GPU, internet. It also sends model downloads to `/tmp`, so the ~8 GB model doesn't fill the ~20 GB working folder or get saved with every version.
**Files used:** none.

In [ ]:
import os
import shlex
import shutil
import socket
import sys

from IPython import get_ipython


def run_or_stop(command: str) -> None:
    """Run a terminal command, show its output, and stop the notebook with a clear message if it fails."""
    get_ipython().system(command)
    # Here we read the command's exit code: 0 means it worked.
    exit_code = get_ipython().user_ns.get("_exit_code", 0)
    if exit_code != 0:
        raise RuntimeError(f"This step failed (exit code {exit_code}): {command}\nSee the output above.")


# Here we work out which machine this is: Kaggle and Colab each leave a recognisable mark.
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules
# Here we pick the folder we may write in on that machine, and put the repo inside it.
WORKING_DIR = "/kaggle/working" if IN_KAGGLE else "/content" if IN_COLAB else os.getcwd()
REPO_DIR = os.path.join(WORKING_DIR, "Is-the-Judge-the-Bottleneck")
QUOTED_REPO_DIR = shlex.quote(REPO_DIR)   # safe to put in a terminal command even if the path has spaces
# Here we check for a GPU: the nvidia-smi tool only exists on machines that have one.
HAS_GPU = shutil.which("nvidia-smi") is not None

# Here we keep model downloads in /tmp on Kaggle and Colab: not saved with the notebook, re-downloaded in ~1 min.
if IN_KAGGLE or IN_COLAB:
    os.environ["HF_HOME"] = "/tmp/huggingface"

print("Machine:", "Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "local")
print("Python: ", sys.version.split()[0], "(the Mac uses 3.12, Kaggle 3.13; both give identical results)")
print("GPU:    ", "yes" if HAS_GPU else "none (set Accelerator to GPU T4 x2 for sections 9-11)")
if HAS_GPU:
    run_or_stop("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

# Here we check the internet by looking up github.com; without it nothing can be downloaded.
try:
    socket.gethostbyname("github.com")
    print("Internet: on")
except OSError:
    raise SystemExit("No internet. On Kaggle: verify your phone number (kaggle.com > Settings), "
                     "then switch Internet on in the notebook's settings panel and run all again.")

## 3. Get the project code

**What it does:** downloads the code of branch `BRANCH` from GitHub. If it's already here (re-running in the same session), it updates to the newest commit on `BRANCH`, throwing away any stray local edits so the code always matches GitHub exactly.
**Files used:** the GitHub repository `shoimya/Is-the-Judge-the-Bottleneck`.

In [ ]:
if os.path.isdir(REPO_DIR):
    # Here we update an existing copy to exactly what GitHub has on BRANCH.
    run_or_stop(f"git -C {QUOTED_REPO_DIR} fetch origin {BRANCH}")
    run_or_stop(f"git -C {QUOTED_REPO_DIR} checkout --force -B {BRANCH} origin/{BRANCH}")
else:
    run_or_stop(f"git clone --branch {BRANCH} {REPO_URL} {QUOTED_REPO_DIR}")

# Here we move into the repo folder, so every later command runs from there.
os.chdir(REPO_DIR)
run_or_stop("git log -1 --oneline")   # which commit this session runs (it is also written into every run log)

## 4. Install the project's Python libraries

**What it does:** installs the exact library versions the project needs, and installs the repo's own `pipeline/` folder (the `-e .` line) so every script can use it.
**Files used:** `requirements.txt` and `pyproject.toml` from the repository.
**Expected:** a long list of yellow "dependency conflicts" about Kaggle's own packages. We don't use those; ignore it.

In [ ]:
run_or_stop("pip install -q -r requirements.txt")

## 5. Keep run logs safe

**What it does:** points the project's `runs/` folder at storage that survives: `/kaggle/working/runs` on Kaggle (saved when you click *Save Version*), Google Drive on Colab. Every run of the pipeline writes its log there, in one folder per scenario: `runs/<scenario>/<start time>/meta.json` and `questions.jsonl`.
**Files used:** none yet; section 11 writes the first log.

In [ ]:
# Here we choose where run logs are really stored.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    runs_storage = "/content/drive/MyDrive/Is-the-Judge-the-Bottleneck/runs"
else:
    runs_storage = os.path.join(WORKING_DIR, "runs")
os.makedirs(runs_storage, exist_ok=True)

# Here we make the repo's runs/ a shortcut (symlink) to that storage, unless it already is one.
if not os.path.islink("runs"):
    shutil.rmtree("runs", ignore_errors=True)   # an empty runs/ folder, if there is one
    os.symlink(runs_storage, "runs")
print("runs/ ->", os.path.realpath("runs"))

## 6. Setup check

**What it does:** runs the project's own check: Python version, the settings file, and that `runs/` can be written to. It must end with **`Setup OK.`**
**Files used:** `run.py` and `config.yaml`.

In [ ]:
# Here we call run.py's setup check directly (the scripts take no command-line options).
run_or_stop('python -c "from run import check_setup; check_setup()"')

## 7. Question sets (T02)

**What it does:** downloads HotpotQA's dev questions and picks the same 100 Pilot and 1,000 Test questions as on the Mac (fixed seed), then checks the picked ids match the lists saved on GitHub.
**Files used:** downloaded from Hugging Face (`hotpotqa/hotpot_qa`, 28 MB) into `data/hotpotqa/`; compared with `results/question_ids/`. Script: `setup/get_questions.py`.

In [ ]:
run_or_stop("python setup/get_questions.py")
# Here we compare the picked ids with GitHub's; any difference stops the notebook.
run_or_stop("git diff --exit-code --stat results/question_ids")
print("Same 100 + 1,000 questions as on the Mac.")

## 8. Search index (T03)

**What it does:** connects our saved BM25 search index to the project, then searches with the 100 Pilot questions and checks the results match the Mac's (11 / 24 / 34 / 44 % "all Gold paragraphs found" in the top 2 / 5 / 10 / 20).
**Files used:** the attached Kaggle dataset `hotpotqa-bm25-index` (7 files, ~2.8 GB, read-only under `/kaggle/input/`), linked as `data/wiki/bm25_index`; compared with `results/pilot/bm25_recall.csv`. Scripts: `setup/check_index.py` (and `setup/build_index.py` only if the dataset is lost: set `BUILD_INDEX_IF_MISSING = True` in section 1).

In [ ]:
import glob

INDEX_LINK = "data/wiki/bm25_index"

# Here we remove a broken link (its target is gone, e.g. the dataset was detached), so it can be linked again.
if os.path.islink(INDEX_LINK) and not os.path.exists(INDEX_LINK):
    os.remove(INDEX_LINK)

# Here we look for the index among the datasets attached to this notebook.
attached_index_matches = glob.glob("/kaggle/input/**/bm25_index/params.index.json", recursive=True)

# Here we make the index available where the code expects it:
#   already linked earlier this session -> nothing to do
#   attached as a dataset               -> link to it (no copying)
#   missing, rebuild allowed            -> build it from scratch
#   missing                             -> stop and say how to attach it
if os.path.exists(INDEX_LINK):
    print("Index already linked.")
elif attached_index_matches:
    os.makedirs("data/wiki", exist_ok=True)
    os.symlink(os.path.dirname(attached_index_matches[0]), INDEX_LINK)
    print("Linked the attached index:", os.path.dirname(attached_index_matches[0]))
elif BUILD_INDEX_IF_MISSING:
    run_or_stop("python setup/build_index.py")   # resumes by itself if the download is interrupted
    run_or_stop("rm -rf data/wiki/dump")          # keep only the index in the saved output
else:
    raise SystemExit("Search index not found. Attach the hotpotqa-bm25-index dataset "
                     "(Add Input > Your Datasets) and re-run this cell.")

# Here we measure search quality and compare it with the Mac's saved numbers.
run_or_stop("python setup/check_index.py")
run_or_stop("git diff --exit-code results/pilot/bm25_recall.csv")
print("Search results match the Mac.")

## 9. Install vLLM (T04, GPU only)

**What it does:** installs vLLM, the program that runs the model on the GPU, then removes TorchAudio: Kaggle preinstalls it built for a different CUDA version, and it crashes vLLM (we never use audio). Finally it shows the installed versions and checks PyTorch can see the GPU.
**Files used:** `requirements-gpu.txt` (vLLM 0.30.0, ~5–10 min to install).

In [ ]:
if not RUN_MODEL_CHECKS:
    print("Skipped (RUN_MODEL_CHECKS is False).")
elif not HAS_GPU:
    raise SystemExit("No GPU. Set Accelerator to GPU T4 x2 in Session options, then run all again.")
else:
    run_or_stop("pip install -q -r requirements-gpu.txt")
    run_or_stop("pip uninstall -y -q torchaudio")
    # Here we list the versions that matter, and check PyTorch sees the GPU.
    run_or_stop("pip list 2>/dev/null | grep -E '^(vllm|torch|transformers|huggingface_hub) '")
    run_or_stop("python -c 'import torch; print(\"PyTorch sees the GPU:\", torch.cuda.is_available())'")

## 10. Model check: smoke test + speed benchmark (T04, GPU only)

**What it does:** loads the model once, then:
- **smoke test:** one tiny question per role (answerer, rewriter, judge), printing each answer and the model's top alternatives for its last word (the judge's line shows how sure it was of "no" vs "yes");
- **benchmark:** 20 Pilot questions through a practice 3-Round loop with real search results. The last lines are the speed and an upper-bound GPU-hour estimate: copy them into the Throughput section of `NOTES.md`.

It shuts the model down at the end, so the cell finishes by itself (~5–10 min in total).
**Files used:** the model `Qwen/Qwen3-4B-Instruct-2507` (~8 GB, downloaded into `/tmp`), the Pilot questions and the index. Script: `setup/check_model.py`.

In [ ]:
if RUN_MODEL_CHECKS:
    # Here we run check_model.py's function with the GPU backend and the benchmark switched on.
    run_or_stop('python -c "from setup.check_model import check_model; '
                'check_model(backend=\'vllm\', include_benchmark=True)"')

## 11. Run the pipeline and see its log (GPU only)

**What it does:** runs `SCENARIO` from section 1 on the GPU, through the same `run.py` you use on the Mac, and logs it. By default that's Single-turn on the first Pilot question: search with the question, then the answerer answers from what came back. Then it shows the log it wrote: what was run (`meta.json`) and, for the first question, the search query, every paragraph found, and the answer (`questions.jsonl`).

If the session ends partway through a whole question set, run this cell again: the run continues where it stopped.
**Files used:** the questions (section 7), the index (section 8), the model (section 10, reused from `/tmp`). Writes `runs/<scenario>/<start time>/`.

In [ ]:
import json


def newest_run_folder(runs_folder: str = "runs") -> str:
    """The run folder whose meta.json was written most recently (the run that just finished)."""
    meta_files = glob.glob(os.path.join(runs_folder, "*", "*", "meta.json"))
    return os.path.dirname(max(meta_files, key=os.path.getmtime))


def show_run_log(run_folder: str) -> None:
    """Print what the run was (meta.json) and what happened to its first question (questions.jsonl)."""
    with open(os.path.join(run_folder, "meta.json")) as meta_file:
        meta = json.load(meta_file)
    print("Log folder:", run_folder)
    for field_name in ["scenario", "question_set", "started_at", "finished_at", "backend", "models", "git_commit"]:
        print(f"  {field_name}: {meta[field_name]}")

    with open(os.path.join(run_folder, "questions.jsonl")) as questions_file:
        records = [json.loads(line) for line in questions_file]
    first_record = records[0]
    print(f"\n{len(records)} question(s) logged. The first one:")
    print("  question:   ", first_record["question"])
    print("  gold answer:", first_record["gold_answer"])
    # Here we show each Round: what was searched, what came back, what the answerer saw.
    for single_round in first_record["rounds"]:
        if single_round["retriever"] is not None:
            print(f"  Round {single_round['round']} search:", single_round["retriever"]["query"])
            for paragraph in single_round["retriever"]["retrieved"]:
                print(f"     [{paragraph['score']:.1f}] {paragraph['title']}: {paragraph['text'][:80]}...")
        print("  answerer saw:", single_round["answerer"]["paragraphs_seen"])
    print("  answer:     ", first_record["final_answer"])


if RUN_MODEL_CHECKS:
    # Here we turn section 1's choices into run.py's settings: no QUESTION_SET means the first Pilot question.
    run_whole_set = QUESTION_SET is not None
    question_set_name = QUESTION_SET or "pilot"
    scenario_call = f"run_one_scenario({SCENARIO!r}, {question_set_name!r}, None, {run_whole_set}, 'vllm')"
    run_or_stop(f'python -c "from run import run_one_scenario; {scenario_call}"')
    show_run_log(newest_run_folder())
    print("\nTo keep this log, click Save Version (it saves /kaggle/working/runs).")